# Agent state: write and read

**Module:** Module 2 — Advanced Agent Patterns  
**Notebook:** `notebooks/module-2/2.2_state.ipynb`

## Purpose
Extend agent state, update it from a tool, and read persisted state in later steps of an execution.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- MCP standardizes how an application exposes tools and resources to clients.
- Context is for runtime dependencies; state is for data that belongs to the agent execution/thread.
- Multi-agent design works best when each specialist has a narrow responsibility and clear input/output.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()

### Implementation step 2

Imports the libraries and APIs needed by the following example: from langchain.agents import AgentState. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import AgentState. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.tools import tool, ToolRuntime; from langgraph.types import Command; from langchain.messages import ToolMessage. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.tools import tool, ToolRuntime; from langgraph.types import Command; from langchain.messages import ToolMessage. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations.

from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

### Implementation step 5

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

### Implementation step 6

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

### Implementation step 7

Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

from pprint import pprint

pprint(response)

### Implementation step 8

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

## Read state

### Implementation step 10

Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

### Implementation step 11

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

### Implementation step 12

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)